#Embeddings

In deep learning, we work with vectors. But how do you represent discrete items like words from a vocabulary? A one-hot encoding (a vector of all zeros except for a single 1) is a possibility, but it's inefficient for large vocabularies and doesn't capture any relationships between words.

**Embeddings** solve this by mapping each discrete token (like a word) to a dense, continuous, and lower-dimensional vector. These vectors are learned during training, and their positions in the vector space can capture semantic relationships (e.g., the vectors for 'king' and 'queen' might be close to each other).

![embeddinds](https://arize.com/wp-content/uploads/2022/06/blog-king-queen-embeddings.jpg)

PyTorch's `nn.Embedding` layer is essentially a lookup table where each token's integer index corresponds to its learned vector.

In [ ]:
import torch
import torch.nn as nn

# Let's imagine a vocabulary of 10 words.
vocab_size = 10
# We want to represent each word with a 3-dimensional vector.
embedding_dim = 3

# Create the embedding layer
embedding_layer = nn.Embedding(vocab_size, embedding_dim)

# Let's represent a sentence with integer indices. e.g., "the cat sat"
word_indices = torch.tensor([0, 1, 2], dtype=torch.long)

# Get the embeddings for these words
word_embeddings = embedding_layer(word_indices)

print(f"Shape of word indices: {word_indices.shape}")
print(f"Shape of word embeddings: {word_embeddings.shape}")
print("\nEmbeddings:\n", word_embeddings)

Shape of word indices: torch.Size([3])
Shape of word embeddings: torch.Size([3, 3])

Embeddings:
 tensor([[-0.2721,  0.8983,  0.3697],
        [-0.2325, -0.3181,  0.1432],
        [ 1.1715,  0.6092, -1.7618]], grad_fn=<EmbeddingBackward0>)


# Assignment: Text Generation

Your assignment is to build and train a character-level recurrent neural network. The goal is to train the network on a large text file (Leo Tolstoy's *Anna Karenina*) and then use it to generate new text that mimics the author's style.

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import requests

# Download the text file
url = 'https://www.gutenberg.org/files/1399/1399-0.txt'
response = requests.get(url)
response.encoding = 'utf-8-sig' # Handle potential BOM
text = response.text
print("Text file downloaded.")

# Create character dictionaries
chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}
encoded = np.array([char2int[ch] for ch in text])

# Function to create batches
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr)//batch_size_total
    arr = arr[:n_batches * batch_size_total]
    arr = arr.reshape((batch_size, -1))
    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

Text file downloaded.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, tokens, n_hidden=512, n_layers=2, drop_prob=0.5, embedding_dim=256):
        super().__init__()
        self.n_layers = n_layers
        self.n_hidden = n_hidden
        self.embedding_dim = embedding_dim

        self.vocab_size = len(tokens)
        self.char2int = {ch: ii for ii, ch in enumerate(tokens)}
        self.int2char = dict(enumerate(tokens))

        # Add an embedding layer
        self.embedding = nn.Embedding(self.vocab_size, embedding_dim)

        # The LSTM now takes the embedding dimension as input
        self.lstm = nn.LSTM(embedding_dim, n_hidden, n_layers,
                            dropout=drop_prob, batch_first=True)

        # Add Dropout
        self.dropout = nn.Dropout(drop_prob)

        # Add a fully connectd layer (hidden state, vocab size)
        self.fc = nn.Linear(n_hidden, self.vocab_size)

    def forward(self, x, hidden):

        # The input x (integer indices) is passed through the embedding layer first
        embeds = self.embedding(x)

        # The rest of the forward pass remains the same as usual
        r_output, hidden = self.lstm(embeds, hidden)
        out = self.dropout(r_output)
        out = out.contiguous().view(-1, self.n_hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        weight = next(self.parameters()).data
        hidden = (weight.new(self.n_layers, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layers, batch_size, self.n_hidden).zero_())
        return hidden

In [8]:
def train(net, data, epochs=20, batch_size=10, seq_length=50, lr=0.001, clip=5):
    net.train()
    # Define an optimizer for the net.parameters()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    # Define the loss function
    criterion = nn.CrossEntropyLoss()

    for e in range(epochs):
        h = net.init_hidden(batch_size)
        for i, (x, y) in enumerate(get_batches(data, batch_size, seq_length)):
            # x is now directly used as a tensor of integer indices.
            inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

            h = tuple([each.data for each in h])

            # Set gradients to zero before performing backprop
            opt.zero_grad()
            output, h = net(inputs, h)
            loss = criterion(output, targets.view(batch_size*seq_length).long())
            # Perform backprop to get the new gradients
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            # Update weights using step
            opt.step()

            # Print the loss every 5 epochs
            if (e + 1) % 5 == 0:
              print(f"Epoch: {e+1}/{epochs}... Loss: {loss.item():.4f}")

model = CharRNN(chars)
print(model)
train(model, encoded, epochs=20, batch_size=128, seq_length=100)

CharRNN(
  (embedding): Embedding(98, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=98, bias=True)
)
Epoch: 5/20... Loss: 1.4506
Epoch: 5/20... Loss: 1.4027
Epoch: 5/20... Loss: 1.4031
Epoch: 5/20... Loss: 1.4130
Epoch: 5/20... Loss: 1.4017
Epoch: 5/20... Loss: 1.3745
Epoch: 5/20... Loss: 1.4000
Epoch: 5/20... Loss: 1.3881
Epoch: 5/20... Loss: 1.4097
Epoch: 5/20... Loss: 1.3405
Epoch: 5/20... Loss: 1.3643
Epoch: 5/20... Loss: 1.4002
Epoch: 5/20... Loss: 1.3833
Epoch: 5/20... Loss: 1.4006
Epoch: 5/20... Loss: 1.4065
Epoch: 5/20... Loss: 1.3694
Epoch: 5/20... Loss: 1.3751
Epoch: 5/20... Loss: 1.3979
Epoch: 5/20... Loss: 1.4145
Epoch: 5/20... Loss: 1.3878
Epoch: 5/20... Loss: 1.3900
Epoch: 5/20... Loss: 1.3950
Epoch: 5/20... Loss: 1.3915
Epoch: 5/20... Loss: 1.4008
Epoch: 5/20... Loss: 1.3563
Epoch: 5/20... Loss: 1.3623
Epoch: 5/20... Loss: 1.3518
Epoch: 5/20... Loss: 1.370

In [9]:
def sample(net, size, prime='The', top_k=None):
    # .eval() is a kind of switch for some specific layers/parts of the model that behave differently during training and inference
    # (evaluating) time. For example, Dropouts Layers, BatchNorm Layers etc. You need to turn them off during model evaluation, and
    # .eval() will do it for you. Put the network on the evaluation mode:
    net.eval()

    chars = [ch for ch in prime]
    h = net.init_hidden(1)
    for ch in prime:
        char, h = predict(net, ch, h, top_k=top_k)

    chars.append(char)

    for ii in range(size):
        char, h = predict(net, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

def predict(net, char, h, top_k=None):
    x = torch.tensor([[net.char2int[char]]], dtype=torch.long)
    inputs = x

    h = tuple([each.data for each in h])
    out, h = net(inputs, h)

    # Apply softmax
    p = F.softmax(out, dim=1)
    p = p.data

    if top_k is None:
        top_ch = np.arange(net.vocab_size)
    else:
        p, top_ch = p.topk(top_k)
        top_ch = top_ch.numpy().squeeze()

    p = p.numpy().squeeze()
    # handle the case where top_ch is a single integer
    if top_ch.size == 1:
        char = top_ch.item()
    else:
        char = np.random.choice(top_ch, p=p/p.sum())

    return net.int2char[char], h

# Generate some text!
print(sample(model, 1000, prime='Anna', top_k=5))

Anna an and-peasant property.”

“I should be to be as mut to see the care of the steps in the sound.
And
how’s that,” answered Anna, and she had already sent
this transfraction and hands in his face and what he had not been talking to
him.

“I want your parents than the same and at the country, and I have
already been moved in him. He cannot be nothing at the stubbors,
and an infection, then.”

“Well, to think it as your position,” said Levin, addressing
Vronsky, and the correct of his head of the sort of tear with
which the same soft-couths are wearing her, and there was a good onely
story woman there an assembling worldly’s people and a candad
angry with them with the doctor.

“Would me here the matter? He’s always that statements as a loss
of considerable angry.” And with his face was aware of them, and talking
that she had seen any definite more than the memory of so much, at the
soldier to him, although the sound of sportsmen are the conversation
he stopped the wa
